In [ ]:
import logging
from poprawione import InspireClassifier

import warnings
import kagglehub
import numpy as np
from kagglehub import KaggleDatasetAdapter
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


np.random.seed(42)
logging.basicConfig(level=logging.INFO)
warnings.simplefilter("ignore")

In [2]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "uciml/default-of-credit-card-clients-dataset",
    "UCI_Credit_Card.csv",
)

INFO:numexpr.utils:NumExpr defaulting to 8 threads.


In [3]:
df.shape

(30000, 25)

In [4]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [5]:
X = df.drop(columns=["ID", "default.payment.next.month"]).to_numpy()
y = df["default.payment.next.month"].to_numpy()

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42
)

In [7]:
Counter(y_train), Counter(y_val), Counter(y_test)

(Counter({0: 16324, 1: 4676}),
 Counter({0: 3505, 1: 995}),
 Counter({0: 3535, 1: 965}))

In [ ]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=3,  # allows some pattern learning, but still weak
        min_samples_split=5,  # prevent very specific splits
        min_samples_leaf=3,  # reduce variance
        max_features=0.8,  # limit number of features considered at each split
        random_state=42,
    ),
)

model.fit(X_train, y_train, X_val, y_val, approximate_knn_=False, save_history_=True, oversampling_per_step=5000)

INFO:poprawione:Fitting KNN index... (0.22 seconds)
INFO:poprawione:Removing outliers... (0.00 seconds)
INFO:poprawione:Identified minority class: 1
INFO:poprawione:Fitting KNN index for validation set... (0.12 seconds)
INFO:poprawione:Identifying border regions... (0.02 seconds)
INFO:poprawione:Training models (step 0)... (0.05 seconds)
INFO:poprawione:Performing adaptive optimizations (step 1)... (0.01 seconds)
INFO:poprawione:Training models (step 1)... (0.07 seconds)
INFO:poprawione:Performing adaptive optimizations (step 2)... (0.01 seconds)
INFO:poprawione:Training models (step 2)... (0.06 seconds)
INFO:poprawione:Performing adaptive optimizations (step 3)... (0.01 seconds)
INFO:poprawione:Training models (step 3)... (0.05 seconds)
INFO:poprawione:Performing adaptive optimizations (step 4)... (0.01 seconds)
INFO:poprawione:Training models (step 4)... (0.06 seconds)
INFO:poprawione:Performing adaptive optimizations (step 5)... (0.01 seconds)
INFO:poprawione:Training models (step 5

CustomEnsembleClassifier(base_estimator=DecisionTreeClassifier(max_depth=3,
                                                               max_features=0.8,
                                                               min_samples_leaf=3,
                                                               min_samples_split=5,
                                                               random_state=42),
                         n_estimators=100)

In [13]:
print(f"Removed outliers: {np.sum(model.history[0]['removed_mask'])}")
print(f"Border indexes found: {np.sum(model.history[1]['border_mask'])}")

Removed outliers: 1997
Border indexes found: 772


In [15]:
i = 3

while i < len(model.history):
    y_pred = model.history[i]["preds"]
    print(f"\nIteration {int((i - 1) / 2)}:")
    print(classification_report(y_val, y_pred))

    print(f"Problematic indexes found: {np.sum(model.history[i + 1]['bp_mask'])}")
    print(
        f"Problematic overlap with border indexes: {np.sum(model.history[i + 1]['bp_mask'] & model.history[1]['border_mask'])}"
    )

    i += 2


Iteration 1:
              precision    recall  f1-score   support

           0       0.83      0.97      0.89      3505
           1       0.72      0.30      0.43       995

    accuracy                           0.82      4500
   macro avg       0.77      0.63      0.66      4500
weighted avg       0.81      0.82      0.79      4500

Problematic indexes found: 1989
Problematic overlap with border indexes: 73

Iteration 2:
              precision    recall  f1-score   support

           0       0.83      0.97      0.89      3505
           1       0.72      0.30      0.42       995

    accuracy                           0.82      4500
   macro avg       0.77      0.63      0.66      4500
weighted avg       0.80      0.82      0.79      4500

Problematic indexes found: 1992
Problematic overlap with border indexes: 73

Iteration 3:
              precision    recall  f1-score   support

           0       0.85      0.91      0.88      3505
           1       0.60      0.44      0.51

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

INFO:poprawione:Performing predictions... (0.04 seconds)


              precision    recall  f1-score   support

           0       0.85      0.91      0.88      3535
           1       0.55      0.41      0.47       965

    accuracy                           0.80      4500
   macro avg       0.70      0.66      0.67      4500
weighted avg       0.78      0.80      0.79      4500

